## Attention

In [ ]:
import torch
import torch.nn as nn
from math import sqrt

class Attention(nn.Module):

    def __init__(self, hidden_dim, context_size, dropout):
        super().__init__()

        self.k_weights = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.q_weights = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_weights = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.mask = torch.triu(torch.ones((context_size, context_size)), diagonal=1).bool()
    def forward(self, x, casual):

        # x size [batch_size, seq_len, hidden_dim]
        batch_size, seq_len, hidden_dim = x.shape

        q = self.q_weights(x)
        k = self.k_weights(x)
        v = self.v_weights(x)

        # attn_values size [batch_size, seq_len, seq_len]
        attn_values = q @ k.transpose(-2, -1)

        if casual == True:
            mask = self.mask[:seq_len, :seq_len]
            attn_values = torch.softmax(torch.masked_fill(attn_values, mask, -torch.inf) /  sqrt(hidden_dim), dim=-1)
        else:
            attn_values = torch.softmax(attn_values /  sqrt(hidden_dim), dim=-1)

        result = self.dropout(attn_values) @ v

        return result, attn_values

## Multi-Head Attention

In [ ]:
class Multi_Head_Attention(nn.Module):

    def __init__(self, hidden_dim, context_size, dropout, head_nums):
        super().__init__()

        self.k_weights = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.q_weights = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_weights = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.head_size = int(hidden_dim/head_nums)
        self.head_nums = head_nums

        self.dropout = nn.Dropout(dropout)
        self.mask = torch.triu(torch.ones((context_size, context_size)), diagonal=1).bool()


    def forward(self, x, casual):

        # x size [batch_size, seq_len, hidden_dim]
        batch_size, seq_len, hidden_dim = x.shape

        q = self.q_weights(x)
        k = self.k_weights(x)
        v = self.v_weights(x)

        # result shape [batch_size, num_heads, seq_len, head_size] <- [batch_size, seq_len, num_heads, head_size]
        k = k.view(batch_size, seq_len, self.head_nums, self.head_size).transpose(1, 2)
        q = q.view(batch_size, seq_len, self.head_nums, self.head_size).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.head_nums, self.head_size).transpose(1, 2)

        # attn_result size [batch_size, head_nums, seq_len, seq_len]
        attn_result = q @ k.transpose(-2, -1)

        if casual == True:
            mask = self.mask[:seq_len, :seq_len]
            attn_values = torch.softmax(torch.masked_fill(attn_result, mask, -torch.inf) / sqrt(self.head_size), dim=-1)
        else:
            attn_values = torch.softmax(attn_values / sqrt(self.head_size), dim=-1)

        # result shape [batch_size, seq_len, num_heads, head_size] <- [batch_size, num_heads, seq_len, head_size]
        result = (self.dropout(attn_result) @ v).transpose(1, 2)

        # result shape [batch_size, seq_len, num_heads * head_size]
        result = result.contiguous().view(batch_size, seq_len, self.head_nums * self.head_size)

        return result, attn_values

## Final Outputs

In [50]:
import tiktoken
from recap import Embeddings

tokenizer = tiktoken.get_encoding("cl100k_base")
emb_dim = 512
embedding = Embeddings(vocab_size=len(tokenizer._mergeable_ranks), context_size=256, emb_dim=emb_dim)
attention = Attention(hidden_dim=emb_dim, context_size=256, dropout=0.2)
multi_attention = Multi_Head_Attention(hidden_dim=emb_dim, context_size=256, dropout=0.2, head_nums=8)

In [52]:
text = "Hello, do you like tea?"
text_tensor = torch.tensor(tokenizer.encode(text)).unsqueeze(0)
output_encoding = embedding(text_tensor)

In [53]:
result, attn_values = attention(output_encoding, True)
print(attn_values.shape)
print(attn_values[0])

torch.Size([1, 7, 7])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7613, 0.2387, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4081, 0.2159, 0.3760, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5771, 0.0922, 0.2001, 0.1305, 0.0000, 0.0000, 0.0000],
        [0.4154, 0.0855, 0.1754, 0.1845, 0.1393, 0.0000, 0.0000],
        [0.3793, 0.0974, 0.1018, 0.1366, 0.1549, 0.1301, 0.0000],
        [0.2815, 0.1007, 0.0845, 0.1575, 0.1692, 0.0882, 0.1184]],
       grad_fn=<SelectBackward0>)


In [55]:
result, attn_values = multi_attention(output_encoding, True)
print(attn_values.shape)
print(attn_values[0, :2])

torch.Size([1, 8, 7, 7])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5761, 0.4239, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3666, 0.2063, 0.4271, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3890, 0.2759, 0.2682, 0.0669, 0.0000, 0.0000, 0.0000],
         [0.3252, 0.1785, 0.2277, 0.1871, 0.0815, 0.0000, 0.0000],
         [0.2259, 0.0820, 0.1493, 0.1768, 0.1503, 0.2158, 0.0000],
         [0.1901, 0.1276, 0.1855, 0.1164, 0.0294, 0.2451, 0.1057]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4506, 0.5494, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2816, 0.3749, 0.3436, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1384, 0.2595, 0.2819, 0.3201, 0.0000, 0.0000, 0.0000],
         [0.1651, 0.1450, 0.1590, 0.2089, 0.3220, 0.0000, 0.0000],
         [0.1027, 0.1913, 0.2361, 0.1502, 0.2167, 0.1031, 0.0000],
         [0.1234, 0.1011, 0.2878, 0.2240, 0.1214, 0.0574, 0.0850]]],
       grad_fn=<SliceBackward0>)
